# QIAS RAG - Minimal Colab (HF / Unsloth)

Minimal Google Colab workflow for end-to-end smoke testing.

- Backend: `huggingface` (default) or `unsloth` (optional GPU path)
- No Ollama usage in this notebook


In [1]:
# 1) GPU check
!nvidia-smi

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Wed Apr 15 09:28:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   27C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [2]:
# 2) Mount Drive and select backend
from google.colab import drive
drive.mount('/content/drive')

CLIENT_TYPE = "huggingface"  # choose: "huggingface" or "unsloth"
assert CLIENT_TYPE in {"huggingface", "unsloth"}
print("Selected backend:", CLIENT_TYPE)

# Main project repo (this one is required)
REPO_URL = "https://github.com/swaileh/qias-mawarith-rag.git"
REPO_DIR = "/content/qias_rag_thinking"

# Optional shared-task dataset repo. Leave as None to skip cloning.
SHARED_TASK_URL = None
SHARED_TASK_DIR = "/content/qias_shared_task_2026"

PDF_DIR = "/content/drive/MyDrive/QIAS26/qias_rag_thinking/data/pdfs"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Selected backend: huggingface


In [3]:
# 3) Clone/update repos (rerun-safe)
import os
import shutil
import subprocess

FORCE_REFRESH = False  # set True to delete and re-clone repositories

if FORCE_REFRESH and os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Using existing repo: {REPO_DIR}")

# Optional shared-task dataset repo (skipped by default)
if SHARED_TASK_URL:
    if FORCE_REFRESH and os.path.exists(SHARED_TASK_DIR):
        shutil.rmtree(SHARED_TASK_DIR)
    if not os.path.exists(SHARED_TASK_DIR):
        print(f"Attempting to clone optional shared-task repo from: {SHARED_TASK_URL}")
        clone_cmd = [
            "git", "-c", "credential.helper=", "-c", "core.askPass=", "-c", "credential.interactive=false",
            "clone", SHARED_TASK_URL, SHARED_TASK_DIR,
        ]
        res = subprocess.run(clone_cmd, capture_output=True, text=True)
        if res.returncode == 0:
            print("Cloned shared-task repo successfully")
        else:
            print("Could not clone shared-task repo. Continuing without it.")
            if res.stderr:
                print("git stderr:", res.stderr.strip().splitlines()[-1])
    else:
        print(f"Using existing shared-task repo: {SHARED_TASK_DIR}")
else:
    print("Skipping optional shared-task clone (SHARED_TASK_URL is None)")

%cd /content/qias_rag_thinking


Using existing repo: /content/qias_rag_thinking
Skipping optional shared-task clone (SHARED_TASK_URL is None)
/content/qias_rag_thinking


In [5]:
# 4) Install dependencies
!pip install -q -U pip setuptools wheel

# Keep Colab base environment stable, and satisfy ipython runtime requirement.
!pip install -q "jedi>=0.16,<0.20"

!pip install -q -e ".[dev]"
!pip install -q bitsandbytes accelerate

if CLIENT_TYPE == "unsloth":
    !pip install -q unsloth
    print("Installed Unsloth dependencies")
else:
    # Use a known stable Transformers version compatible with Qwen3.5 + Colab torch.
    !pip install -q --upgrade "transformers==5.2.0"
    !pip install -q -U "huggingface_hub>=0.25.0"
    print("Installed HuggingFace dependencies (transformers==5.2.0 with compatible tokenizers)")

print("Note: Colab may still show resolver warnings for unrelated preinstalled packages.")
print("If you previously used a different transformers build, restart runtime once then rerun from cell 3.")


  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for qias-mawarith-rag (pyproject.toml) ... done
Installed HuggingFace dependencies (transformers==5.2.0 with compatible tokenizers)
Note: Colab may still show resolver warnings for unrelated preinstalled packages.
If you previously used a different transformers build, restart runtime once then rerun from cell 3.


In [9]:
# 5) Configure backend and init pipeline
import sys
import yaml
from pathlib import Path

import transformers
print("Transformers version:", transformers.__version__)

# Keep thinking enabled for QIAS answer format, but reduce other latency costs.
SPEED_MODE = False
REQUIRE_THINKING = True

config_path = Path("/content/qias_rag_thinking/config/rag_config.yaml")
with config_path.open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

config["model"]["client_type"] = CLIENT_TYPE
config["model"]["context_window"] = 8192
config["model"]["enable_thinking"] = REQUIRE_THINKING
config.setdefault("evaluation", {})["enable_relevance_evaluation"] = False

if SPEED_MODE:
    # Faster than 1024 while still allowing useful <think> + <answer>
    config["model"]["max_new_tokens"] = 512
    config.setdefault("retrieval", {})["top_k"] = 3
    config["retrieval"]["rerank"] = False
else:
    config["model"]["max_new_tokens"] = 1024

with config_path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, allow_unicode=True, sort_keys=False)

sys.path.insert(0, "/content/qias_rag_thinking")
from qias_mawarith_rag.pipeline import RAGPipeline

pipeline = RAGPipeline(config_path=str(config_path))
print("Pipeline initialized with backend:", pipeline.config["model"]["client_type"])
print("Speed mode:", SPEED_MODE)
print("Require thinking:", REQUIRE_THINKING)
print("max_new_tokens:", pipeline.config["model"].get("max_new_tokens"))
print("enable_thinking:", pipeline.config["model"].get("enable_thinking"))
print("retrieval.top_k:", pipeline.config.get("retrieval", {}).get("top_k"))
print("retrieval.rerank:", pipeline.config.get("retrieval", {}).get("rerank"))


Transformers version: 5.2.0
Initializing RAG Pipeline...
Loading embedding model: sentence-transformers/paraphrase-multilingual-mpnet-base-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector store initialized: 65 documents
Web search is disabled in configuration
Reranking disabled
Using HuggingFace Transformers for Qwen3.5...
Loading Qwen/Qwen3.5-9B from HuggingFace...
Thinking mode: Enabled
🔄 Loading model with eager attention implementation...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

✅ Model loaded with eager attention
✓ Model loaded: Qwen/Qwen3.5-9B
BM25 index loaded from data/bm25_index.pkl (366 docs)
RAG Pipeline initialized successfully
Pipeline initialized with backend: huggingface
Speed mode: False
Require thinking: True
max_new_tokens: 1024
enable_thinking: True
retrieval.top_k: 3
retrieval.rerank: False


In [10]:
# 6) Build/load knowledge base
import os

if os.path.exists(PDF_DIR):
    pdf_files = [f for f in os.listdir(PDF_DIR) if f.lower().endswith('.pdf')]
    print("PDF files found:", len(pdf_files))
    if pdf_files:
        pipeline.build_knowledge_base(PDF_DIR)
        stats = pipeline.vector_store.get_collection_stats()
        print("KB docs:", stats.get("total_documents", 0))
    else:
        print("No PDFs found. Upload PDFs to:", PDF_DIR)
else:
    print("PDF directory missing:", PDF_DIR)


PDF files found: 1

[SKIP] PDF KB already ingested on 2026-04-15T09:32:57.910242+00:00 (65 chunks).
       Pass force_rebuild=True to re-ingest.
KB docs: 65


In [12]:
# 7) One-query latency smoke benchmark
import time

question = "مات وترك: أم و أب و بنت. ما هو نصيب كل وريث؟"
t0 = time.perf_counter()
result = pipeline.query(question, top_k=pipeline.config.get("retrieval", {}).get("top_k", 3))
latency_s = time.perf_counter() - t0

parsed = result.get("parsed_output", {})
thinking_text = parsed.get("thinking", "")

print(f"Latency: {latency_s:.2f}s")
print("Parsing success:", parsed.get("parsing_success"))
print("Validation success:", parsed.get("validation_success"))
print("Retrieved docs:", len(result.get("retrieved_docs", [])))
print("Output chars:", len(result.get("raw_output", "")))
print("Thinking chars:", len(thinking_text))
print("Thinking captured:", len(thinking_text) > 0)



=== Processing Question ===
مات وترك: أم و أب و بنت. ما هو نصيب كل وريث؟...
Retrieving documents...
Reranking documents...
Building prompt...
Generating response with Qwen3...
Generated 2504 chars in 66.6s
Parsing output...
✓ Parsing success: True
✓ Validation success: True
Latency: 66.59s
Parsing success: True
Validation success: True
Retrieved docs: 3
Output chars: 2504
Thinking chars: 1444
Thinking captured: True


In [13]:
# 8) Stage-by-stage profiling (required before further tuning)
import time
import statistics

PROFILE_RUNS = 3
PROFILE_QUESTION = "مات وترك: أم و أب و بنت. ما هو نصيب كل وريث؟"
TOP_K = pipeline.config.get("retrieval", {}).get("top_k", 3)

stage_records = []

for run_idx in range(PROFILE_RUNS):
    print(f"\n--- Profiling run {run_idx+1}/{PROFILE_RUNS} ---")
    rec = {}

    t0 = time.perf_counter()
    docs = pipeline.retriever.retrieve(PROFILE_QUESTION, top_k=TOP_K)
    rec["retrieve_s"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    docs = pipeline.reranker.rerank(PROFILE_QUESTION, docs, top_k=TOP_K)
    rec["rerank_s"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    prompt = pipeline.prompt_builder.build_prompt(
        PROFILE_QUESTION,
        retrieved_docs=docs,
        web_search_results=None,
    )
    rec["prompt_s"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    raw_output = pipeline.qwen_client.generate(prompt)
    rec["generate_s"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    parsed = pipeline.output_parser.parse(raw_output)
    rec["parse_s"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    if parsed.get("thinking"):
        _ = pipeline.thinking_extractor.quality_score(
            parsed["thinking"],
            parsed.get("structured_output"),
        )
        rec["thinking_quality_s"] = time.perf_counter() - t0
    else:
        rec["thinking_quality_s"] = 0.0

    rec["total_s"] = (
        rec["retrieve_s"]
        + rec["rerank_s"]
        + rec["prompt_s"]
        + rec["generate_s"]
        + rec["parse_s"]
        + rec["thinking_quality_s"]
    )
    rec["retrieved_docs"] = len(docs)
    rec["output_chars"] = len(raw_output)
    rec["thinking_chars"] = len(parsed.get("thinking", ""))
    rec["parsing_success"] = bool(parsed.get("parsing_success"))
    rec["validation_success"] = bool(parsed.get("validation_success"))

    stage_records.append(rec)

# Aggregate
keys = [
    "retrieve_s", "rerank_s", "prompt_s", "generate_s", "parse_s", "thinking_quality_s", "total_s"
]
summary = {k: statistics.mean(r[k] for r in stage_records) for k in keys}

print("\n=== Stage Profiling Summary (mean over runs) ===")
for k in keys:
    v = summary[k]
    pct = (v / summary["total_s"] * 100.0) if summary["total_s"] > 0 else 0.0
    print(f"{k:18s}: {v:8.3f}s ({pct:6.2f}%)")

print("\n=== Quality / Size Checks (last run) ===")
last = stage_records[-1]
print("retrieved_docs      :", last["retrieved_docs"])
print("output_chars        :", last["output_chars"])
print("thinking_chars      :", last["thinking_chars"])
print("parsing_success     :", last["parsing_success"])
print("validation_success  :", last["validation_success"])

# Keep summary for next planning cell
profile_summary = {
    "runs": stage_records,
    "mean": summary,
}



--- Profiling run 1/3 ---
Generated 2485 chars in 66.8s

--- Profiling run 2/3 ---
Generated 2593 chars in 67.8s

--- Profiling run 3/3 ---
Generated 2535 chars in 67.4s

=== Stage Profiling Summary (mean over runs) ===
retrieve_s        :    0.007s (  0.01%)
rerank_s          :    0.000s (  0.00%)
prompt_s          :    0.000s (  0.00%)
generate_s        :   67.351s ( 99.99%)
parse_s           :    0.000s (  0.00%)
thinking_quality_s:    0.000s (  0.00%)
total_s           :   67.358s (100.00%)

=== Quality / Size Checks (last run) ===
retrieved_docs      : 3
output_chars        : 2535
thinking_chars      : 2535
parsing_success     : True
validation_success  : True


In [15]:
# 9) Generate exactly 1000 synthetic samples (for KB population)
from pathlib import Path
from qias_mawarith_rag.datagen.generator import FiqhDataGenerator
from qias_mawarith_rag.datagen.exporter import export_to_train_format

SYNTH_TARGET = 100000
SYNTH_DIR = Path("/content/qias_rag_thinking/data/synthetic")
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

gen = FiqhDataGenerator(seed=2026)

n_basic = int(SYNTH_TARGET * 0.50)
n_advanced = int(SYNTH_TARGET * 0.35)
n_edge = SYNTH_TARGET - n_basic - n_advanced

print(f"Generating synthetic set: basic={n_basic}, advanced={n_advanced}, edge={n_edge}")

# unique=False ensures we can reach exact target count quickly
basic = gen.generate_batch(n_basic, difficulty="basic", unique=False)
advanced = gen.generate_batch(n_advanced, difficulty="advanced", unique=False)
edges = gen.generate_edge_cases(n_edge, unique=False)

all_examples = (basic + advanced + edges)[:SYNTH_TARGET]
print("Generated examples:", len(all_examples))

export_to_train_format(all_examples, str(SYNTH_DIR))
print(f"Exported synthetic training files to: {SYNTH_DIR}")

Generating synthetic set: basic=50000, advanced=35000, edge=15000
Generated 50000 examplesttempts: 50000)...
Generated 35000 examplesttempts: 35000)...
Generated examples: 99994
Exported 99994 examples to 1000 files in /content/qias_rag_thinking/data/synthetic
Exported synthetic training files to: /content/qias_rag_thinking/data/synthetic


In [ ]:
# 10) Ingest synthetic samples into KB and rebuild BM25
pipeline.build_knowledge_base_from_synthetic(
    synthetic_directory="/content/qias_rag_thinking/data/synthetic",
    force_rebuild=True,
)

stats = pipeline.vector_store.get_collection_stats()
print("Vector store docs after synthetic ingest:", stats.get("total_documents", 0))


=== Ingesting Synthetic Knowledge Base ===
Source: /content/qias_rag_thinking/data/synthetic
Removing previous synthetic entries from vector store...
  Previous synthetic entries removed.
Found 1000 synthetic data file(s) in /content/qias_rag_thinking/data/synthetic
  Loaded 50/1000 files → 4994 documents so far
  Loaded 100/1000 files → 9994 documents so far
  Loaded 150/1000 files → 14994 documents so far
  Loaded 200/1000 files → 19994 documents so far
  Loaded 250/1000 files → 24994 documents so far
  Loaded 300/1000 files → 29994 documents so far
  Loaded 350/1000 files → 34994 documents so far
  Loaded 400/1000 files → 39994 documents so far
  Loaded 450/1000 files → 44994 documents so far
  Loaded 500/1000 files → 49994 documents so far
  Loaded 550/1000 files → 54994 documents so far
  Loaded 600/1000 files → 59994 documents so far
  Loaded 650/1000 files → 64994 documents so far
  Loaded 700/1000 files → 69994 documents so far
  Loaded 750/1000 files → 74994 documents so far


Batches:   0%|          | 0/3125 [00:00<?, ?it/s]

In [ ]:
# 11) Re-run stage profiling after synthetic KB population
# (same logic as cell 8 for direct before/after comparison)
import time
import statistics

PROFILE_RUNS = 3
PROFILE_QUESTION = "مات وترك: أم و أب و بنت. ما هو نصيب كل وريث؟"
TOP_K = pipeline.config.get("retrieval", {}).get("top_k", 3)

stage_records_after = []

for run_idx in range(PROFILE_RUNS):
    print(f"\n--- Post-ingest profiling run {run_idx+1}/{PROFILE_RUNS} ---")
    rec = {}

    t0 = time.perf_counter()
    docs = pipeline.retriever.retrieve(PROFILE_QUESTION, top_k=TOP_K)
    rec["retrieve_s"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    docs = pipeline.reranker.rerank(PROFILE_QUESTION, docs, top_k=TOP_K)
    rec["rerank_s"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    prompt = pipeline.prompt_builder.build_prompt(
        PROFILE_QUESTION,
        retrieved_docs=docs,
        web_search_results=None,
    )
    rec["prompt_s"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    raw_output = pipeline.qwen_client.generate(prompt)
    rec["generate_s"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    parsed = pipeline.output_parser.parse(raw_output)
    rec["parse_s"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    if parsed.get("thinking"):
        _ = pipeline.thinking_extractor.quality_score(
            parsed["thinking"],
            parsed.get("structured_output"),
        )
        rec["thinking_quality_s"] = time.perf_counter() - t0
    else:
        rec["thinking_quality_s"] = 0.0

    rec["total_s"] = (
        rec["retrieve_s"]
        + rec["rerank_s"]
        + rec["prompt_s"]
        + rec["generate_s"]
        + rec["parse_s"]
        + rec["thinking_quality_s"]
    )
    rec["retrieved_docs"] = len(docs)
    rec["output_chars"] = len(raw_output)
    rec["thinking_chars"] = len(parsed.get("thinking", ""))
    rec["parsing_success"] = bool(parsed.get("parsing_success"))
    rec["validation_success"] = bool(parsed.get("validation_success"))

    stage_records_after.append(rec)

keys = [
    "retrieve_s", "rerank_s", "prompt_s", "generate_s", "parse_s", "thinking_quality_s", "total_s"
]
summary_after = {k: statistics.mean(r[k] for r in stage_records_after) for k in keys}

print("\n=== Post-ingest Stage Profiling Summary (mean over runs) ===")
for k in keys:
    v = summary_after[k]
    pct = (v / summary_after["total_s"] * 100.0) if summary_after["total_s"] > 0 else 0.0
    print(f"{k:18s}: {v:8.3f}s ({pct:6.2f}%)")

print("\n=== Post-ingest Quality / Size Checks (last run) ===")
last_after = stage_records_after[-1]
print("retrieved_docs      :", last_after["retrieved_docs"])
print("output_chars        :", last_after["output_chars"])
print("thinking_chars      :", last_after["thinking_chars"])
print("parsing_success     :", last_after["parsing_success"])
print("validation_success  :", last_after["validation_success"])